# Builder Node — Unit Tests
Step-by-step testing of the Builder Node.
Run each cell independently to inspect intermediate outputs.

**Pre-requisites before running:**
1. `OPENAI_API_KEY` set in `.env`
2. Qdrant running with the `openapi_reference` collection indexed (run `test_rag.ipynb` first)

In [ ]:
# Step 1 — Imports and setup
import json, os
from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE
from nodes.reader import reader_node
from nodes.extractor import extractor_node
from nodes.reflector import reflector_node
from nodes.validator import validator_node
from nodes.builder import builder_node
from utils.parsers import parse_sections
from tools.document_tools import load_markdown
from utils.diagnostic import DiagnosticCollector

logger = get_logger(__name__)

# DiagnosticCollector instantiated directly here — independent of DIAGNOSTIC_MODE in settings,
# so the notebook always collects telemetry regardless of production config.
diagnostic = DiagnosticCollector()
logger.info("Imports OK — DiagnosticCollector ready.")

In [ ]:
# Step 2 — Load document, parse sections, run Reader for context
# parse_sections gives us the sections and reader gives the OpenAPI reference
# context needed by the Extractor. Both are needed before the Planner.
from nodes.reader import reader_node

raw_text = load_markdown(TSPEC_DATA_TEST_FILE)
parsed_sections, excluded_sections_reader = parse_sections(raw_text)

TARGET_IDS = {"325", "346"}

reader_result = reader_node({
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})
logger.info(f"Reader OK — {len(parsed_sections)} sections parsed, {len(excluded_sections_reader)} excluded by reader.")
sections_by_id = {s["section_id"]: s for s in parsed_sections}
test_sections = [sections_by_id[sid] for sid in TARGET_IDS if sid in sections_by_id]
logger.info(f"Test sections: {[s["section_id"] + " " + s["title"][:40] for s in test_sections]}")

In [ ]:
# Step 3 — Run Planner to get the real extraction plan
# Running the Planner verifies that sections_planned and excluded_by_planner
# in the final output match what the Planner actually decided.
# We then limit execution to 2 target sections to keep test cost low.
from nodes.planner import planner_node

planner_result = planner_node({
    **reader_result,
    "parsed_sections":     test_sections,  # only the 2 target sections
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})

# excluded_sections_planner comes from the Planner — reflects its real decisions
excluded_sections_planner = planner_result["excluded_sections_planner"]
full_plan = planner_result["extraction_plan"]

# Limit execution to 2 target sections from the full plan
planned_ids = {s["section_id"] for s in full_plan["sections_to_extract"]}
test_sections_in_plan = [s for s in full_plan["sections_to_extract"] if s["section_id"] in TARGET_IDS]

# If targets not in plan, fall back to first 2 planned sections
if not test_sections_in_plan:
    test_sections_in_plan = full_plan["sections_to_extract"][:2]
    logger.warning(f"Target sections not selected by Planner — using fallback: {[s['section_id'] for s in test_sections_in_plan]}")

extraction_plan = {**full_plan, "sections_to_extract": test_sections_in_plan}

logger.info(f"Planner OK — {len(full_plan['sections_to_extract'])} planned, {len(excluded_sections_planner)} excluded.")
logger.info(f"Test will process: {[s['section_id'] + ' ' + s['title'] for s in test_sections_in_plan]}")

In [ ]:
# Step 4 — Run Extractor on the 2 test sections
extractor_result = extractor_node({
    **reader_result,
    "parsed_sections":     [sections_by_id[s["section_id"]] for s in test_sections_in_plan if s["section_id"] in sections_by_id],
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     extraction_plan,
    "validation_errors":   [],
    "section_feedback":    [],
    "iteration_count":     0,
})
logger.info(f"Extractor OK — {len(extractor_result['raw_rules'])} raw rule(s).")

In [ ]:
# Step 5 — Run Reflector
reflector_result = reflector_node({
    **reader_result,
    **extractor_result,
    "parsed_sections":     [sections_by_id[s["section_id"]] for s in test_sections_in_plan if s["section_id"] in sections_by_id],
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     extraction_plan,
})
logger.info(f"Reflector OK — {len(reflector_result['reflected_rules'])} reflected rule(s).")

In [ ]:
# Step 6 — Run Validator
validator_result = validator_node({
    **reader_result,
    **extractor_result,
    **reflector_result,
    "parsed_sections":     [sections_by_id[s["section_id"]] for s in test_sections_in_plan if s["section_id"] in sections_by_id],
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     extraction_plan,
    "iteration_count":     1,
})
logger.info(f"Validator OK — {len(validator_result['validated_rules'])} validated, {len(validator_result['validation_errors'])} errors.")

# Record iteration 1 in the diagnostic collector
_reflected_1 = reflector_result["reflected_rules"]
diagnostic.record_iteration(
    iteration_num=1,
    reflected_rules=_reflected_1,
    validation_errors=validator_result["validation_errors"],
    validated_rules=validator_result["validated_rules"],
    section_feedback=validator_result.get("section_feedback", []),
    error_rate=len(validator_result["validation_errors"]) / len(_reflected_1) if _reflected_1 else 0.0,
)

In [ ]:
# Step 7 — Loop back up to MAX_ITERATIONS (mirrors real pipeline behavior)
from graph.conditions import should_loop_or_build
from config.settings import MAX_ITERATIONS

all_validated     = list(validator_result["validated_rules"])
current_errors    = validator_result["validation_errors"]
section_feedback  = validator_result.get("section_feedback", [])
current_reflected = reflector_result["reflected_rules"]
iteration_count   = 1

while True:
    decision = should_loop_or_build({
        **reader_result,
        "reflected_rules":   current_reflected,
        "validated_rules":   all_validated,
        "validation_errors": current_errors,
        "iteration_count":   iteration_count,
    })
    logger.info(f"Post-validator decision (iter {iteration_count}): '{decision}'")

    if decision != "extractor":
        break

    ext = extractor_node({
        **reader_result,
        "parsed_sections":     [sections_by_id[s["section_id"]] for s in test_sections_in_plan if s["section_id"] in sections_by_id],
        "main_doc_path":       TSPEC_DATA_TEST_FILE,
        "auxiliary_doc_paths": [],
        "extraction_plan":     extraction_plan,
        "validation_errors":   current_errors,
        "section_feedback":    section_feedback,
        "iteration_count":     iteration_count,
    })
    ref = reflector_node({
        **reader_result, **ext,
        "parsed_sections":     [sections_by_id[s["section_id"]] for s in test_sections_in_plan if s["section_id"] in sections_by_id],
        "main_doc_path":       TSPEC_DATA_TEST_FILE,
        "auxiliary_doc_paths": [],
        "extraction_plan":     extraction_plan,
    })
    iteration_count += 1
    val = validator_node({
        **reader_result, **ext, **ref,
        "parsed_sections":     [sections_by_id[s["section_id"]] for s in test_sections_in_plan if s["section_id"] in sections_by_id],
        "main_doc_path":       TSPEC_DATA_TEST_FILE,
        "auxiliary_doc_paths": [],
        "extraction_plan":     extraction_plan,
        "iteration_count":     iteration_count,
    })
    all_validated    += val["validated_rules"]
    current_errors    = val["validation_errors"]
    section_feedback  = val.get("section_feedback", [])
    current_reflected = ref["reflected_rules"]

    # Record this loop iteration in the diagnostic collector
    _reflected_n = ref["reflected_rules"]
    diagnostic.record_iteration(
        iteration_num=iteration_count,
        reflected_rules=_reflected_n,
        validation_errors=current_errors,
        validated_rules=val["validated_rules"],
        section_feedback=section_feedback,
        error_rate=len(current_errors) / len(_reflected_n) if _reflected_n else 0.0,
    )
    logger.info(f"Iter {iteration_count}: {len(val['validated_rules'])} validated, {len(current_errors)} errors.")

logger.info(f"Total validated going to Builder: {len(all_validated)}")

In [ ]:
# Step 8 — Run Builder
# parsed_sections: full document sections (for correct section accounting).
# The extraction_plan limits actual extraction to the 2 test sections,
# so sections_planned=2 of sections_for_planner=N -> run_scope="partial".
builder_state = {
    **reader_result,
    "main_doc_path":             TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths":       [],
    "extraction_plan":           extraction_plan,
    "validated_rules":           all_validated,
    "validation_errors":         current_errors,
    "iteration_count":           iteration_count,
    "parsed_sections":           parsed_sections,
    "excluded_sections_reader":  excluded_sections_reader,
    "excluded_sections_planner": excluded_sections_planner,
}

builder_result = builder_node(builder_state)

assert "final_output_path" in builder_result
output_path = builder_result["final_output_path"]
assert os.path.isfile(output_path)
logger.info(f"builder_node OK — {output_path}")

In [ ]:
# Step 9 — Inspect the output JSON
with open(output_path, encoding="utf-8") as f:
    rules_bank = json.load(f)

meta    = rules_bank["metadata"]
summary = rules_bank["summary"]
rules   = rules_bank["rules"]

assert set(rules_bank.keys()) == {"metadata", "summary", "rules"}
for field in ("source_document", "generated_at", "model", "total_rules",
              "iterations_run", "sections_total", "sections_for_planner",
              "sections_planned", "sections_with_rules",
              "excluded_by_reader_count", "excluded_by_planner_count",
              "excluded_by_reader", "excluded_by_planner"):
    assert field in meta, f"Missing metadata field: {field}"
assert meta["total_rules"] == len(rules)
assert summary["total_validated"] == len(rules)

logger.info("Output JSON structure OK.")
logger.info("=== METADATA ===")
for k, v in meta.items():
    if not isinstance(v, list):
        logger.info(f"  {k}: {v}")
logger.info(f"  excluded_by_reader  : {meta['excluded_by_reader_count']} section(s)")
logger.info(f"  excluded_by_planner : {meta['excluded_by_planner_count']} section(s)")
logger.info("=== RULES ===")
for i, rule in enumerate(rules, start=1):
    logger.info(f"Rule {i} — [{rule['section_id']}] {rule['rule_type']} / {rule['source_name']}")
    logger.info(f"  rule_text  : {rule['rule_text'][:120]}")
    logger.info(f"  confidence : {rule.get('reflection_confidence', 'n/a')}")
    logger.info(f"  val_passed : {rule.get('validation_passed', 'n/a')}")
    logger.info("")

In [ ]:
# Step 10 — Inspect the diagnostic report
doc_stem  = os.path.splitext(os.path.basename(TSPEC_DATA_TEST_FILE))[0]
diag_path = diagnostic.save(doc_stem)
assert os.path.isfile(diag_path), f"Diagnostic file not found: {diag_path}"
logger.info(f"Diagnostic report saved: {diag_path}")

with open(diag_path, encoding="utf-8") as f:
    diag = json.load(f)

assert "iterations" in diag
for it in diag["iterations"]:
    logger.info(
        f"\n=== Iteration {it['iteration']} — "
        f"{it['total_reflected']} reflected | {it['total_validated']} validated | "
        f"{it['total_errors']} errors | error_rate={it['error_rate']:.1%} ==="
    )
    for rule in it["rules"]:
        v = rule["validator"]
        r = rule["reflector"]
        status = "PASS" if v["passed"] else f"FAIL [{v['error_type']}/{v['stage']}]"
        logger.info(f"  {status}  [{rule['section_id']}] {rule['rule_type']} / {rule['source_name']}")
        if not v["passed"]:
            logger.info(f"    → {v['instruction']}")
        if r["flagged"] or r["split_suggestion"] or r["discard_suggestion"]:
            logger.info(
                f"    reflector: conf={r['confidence']:.2f}  flagged={r['flagged']}  "
                f"split={r['split_suggestion']!r}  discard={r['discard_suggestion']}"
            )
        if r["missing_rules"]:
            logger.info(f"    missing_rules: {r['missing_rules']}")
    if it["section_feedback"]:
        logger.info(f"  section_feedback: {it['section_feedback']}")